In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 04.2 · Transport rebuild and retrain

Notebook de evidencia Day 04.2. Parte de los artefactos cerrados en notebook 12 para responder una sola pregunta: si el bloqueo de `transport_only` venia de parser/join o de ausencia estructural de raw.

Objetivo operativo del dia:
- diagnosticar missingness de transporte,
- construir un staging reconstruido con provenance flags,
- medir cobertura/gate real,
- reentrenar solo si el artefacto final supera cobertura minima y no introduce leakage temporal.

In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT

day041_quality_path = PROJECT_ROOT / "artifacts/public/data_quality_day041_ablation_matrix.json"
day041_summary_path = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306/20260306T095152Z_day041_ablation_run_summary.json"
missingness_json_path = PROJECT_ROOT / "artifacts/public/transport_missingness_day042.json"
missingness_csv_path = PROJECT_ROOT / "artifacts/public/transport_missingness_day042.csv"
day042_quality_path = PROJECT_ROOT / "artifacts/public/data_quality_day042_transport_matrix.json"
day042_summary_path = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306/20260306T_day042_train_day042_transport_run_summary.json"
transport_rebuilt_path = PROJECT_ROOT / "data/public/support/ofertas_transport_signals_day042.csv"

day041_quality = json.loads(day041_quality_path.read_text(encoding="utf-8"))
day041_summary = json.loads(day041_summary_path.read_text(encoding="utf-8"))
missingness = json.loads(missingness_json_path.read_text(encoding="utf-8"))
day042_quality = json.loads(day042_quality_path.read_text(encoding="utf-8"))
day042_summary = json.loads(day042_summary_path.read_text(encoding="utf-8"))
missingness_df = pd.read_csv(missingness_csv_path, keep_default_na=False)
transport_rebuilt_df = pd.read_csv(transport_rebuilt_path, keep_default_na=False)


## 1. Contexto y recap Day 04.1

Day 04.1 dejo dos evidencias clave:
- `transport_only` no podia entrenarse por cobertura (`0.7570` train),
- aun no sabiamos si el hueco venia de parser, mapping o ausencia real de raw comparable.

Day 04.2 se abre solo para aislar esa causa y decidir si merece reabrir modelado de transporte.

In [3]:
display(Markdown("### Recap Day 04.1"))
recap = pd.DataFrame([
    {
        "best_pure_variant_day041": day041_summary["best_pure_variant"],
        "transport_status_day041": day041_summary["transport_variant_status"],
        "transport_gate_pass_day041": day041_summary["transport_gate_pass"],
        "transport_train_coverage_day041": day041_quality["variants"]["transport_only"]["transport_min_train_coverage"],
        "transport_test_coverage_day041": day041_quality["variants"]["transport_only"]["transport_min_test_coverage"],
    }
])
display(recap)


### Recap Day 04.1

,best_pure_variant_day041,transport_status_day041,transport_gate_pass_day041,transport_train_coverage_day041,transport_test_coverage_day041
0,V2_DISPERSION_LR_smote_0.5_v1,excluded_parser_gate_failed,False,0.866663,0.974966


## 2. Diagnostico de missingness y bucketizacion

El script Day 04.2 clasifica cada clave operativa `fecha_evento + producto_canonico + proveedor_candidato` en cinco buckets mutuamente excluyentes:
- `exact_raw_match`
- `parser_fix_possible`
- `deterministic_rebuild_possible`
- `heuristic_same_month_only`
- `no_raw_source`

La heuristica local queda auditada y no se confunde con raw explicito.

In [4]:
display(Markdown("### Bucket counts y cobertura por etapa"))
bucket_counts = pd.DataFrame([
    {
        "bucket_counts_v2_rows": missingness["bucket_counts_v2_rows"],
        "stage_coverage": missingness["stage_coverage"],
    }
])
display(bucket_counts)

display(Markdown("### Buckets por ano (filas V2)"))
year_rows = []
for year, payload in missingness["bucket_breakdown_by_year_v2_rows"].items():
    row = {"year": year}
    row.update(payload)
    year_rows.append(row)
display(pd.DataFrame(year_rows).fillna(0).sort_values("year"))

display(Markdown("### No raw source: subcausas"))
display(pd.DataFrame([missingness["no_raw_subreason_counts_v2_rows"]]))


### Bucket counts y cobertura por etapa

,bucket_counts_v2_rows,stage_coverage
0,"{'exact_raw_match': 137411, 'no_raw_source': 1...",{'raw_explicit': {'coverage_train': 0.86666321...


### Buckets por ano (filas V2)

,year,deterministic_rebuild_possible,exact_raw_match,heuristic_same_month_only,no_raw_source,parser_fix_possible
0,2017,535.0,14866,320,3808,0.0
1,2018,0.0,15036,78,3589,0.0
2,2019,0.0,16488,393,1652,0.0
3,2020,0.0,18707,816,2007,0.0
4,2021,0.0,17608,487,1508,0.0
5,2022,0.0,15652,314,1698,23.0
6,2023,0.0,13951,514,243,0.0
7,2024,0.0,13016,26,93,0.0
8,2025,0.0,12087,157,274,0.0


### No raw source: subcausas

,same_date_product_other_provider_only,no_date_raw,same_date_other_product_only
0,14784,60,28


## 3. Contrato de reconstruccion determinista y heuristica local

Ladder aplicado en Day 04.2:
1. raw explicito de Day 04.1,
2. parser fix por alias/mapping de proveedor,
3. deterministic rebuild desde detalle por terminal del propio raw,
4. heuristica same-month con provenance flags.

Ademas, Day 04.2 audita cuanta parte de esa heuristica mete `lookahead`, porque eso ya no es valido para entrenamiento si usa una fecha futura respecto al evento.

In [5]:
display(Markdown("### Provider aliases recuperados por parser_fix"))
alias_rows = pd.DataFrame(missingness["provider_alias_resolution"]["resolved_aliases"])
display(alias_rows)

display(Markdown("### Detail rebuild summary"))
display(pd.DataFrame([missingness["detail_rebuild_summary"]]))

display(Markdown("### Heuristica same-month y lookahead"))
display(pd.DataFrame([missingness["lookahead_stats"]]))

display(Markdown("### Muestra de claves reconstruidas por heuristica"))
heuristic_sample = missingness_df.loc[
    missingness_df["transport_source_kind"].eq("heuristic_same_month"),
    [
        "fecha_evento",
        "producto_canonico",
        "proveedor_candidato",
        "transport_bucket_origin",
        "transport_days_gap",
        "transport_lookahead_flag",
        "source_date_for_transport",
    ],
].head(20)
display(heuristic_sample)


### Provider aliases recuperados por parser_fix

,proveedor_raw,resolved_provider,alias_method
0,SUPPLIER_061 OIL,SUPPLIER_061,suffix_strip
1,SUPPLIER_019 IBERIA,SUPPLIER_019,suffix_strip
2,SUPPLIER_009 OFERTA,SUPPLIER_009,suffix_strip
3,SUPPLIER_009 PREPAGO,SUPPLIER_009,suffix_strip
4,SUPPLIER_056 ENERGY,SUPPLIER_056,suffix_strip
5,PETRO MEDITERRANEO,SUPPLIER_043,exact_squash
6,RUTA PETROL,SUPPLIER_051,exact_squash
7,PETRO MEDITERRANEA,SUPPLIER_043,fuzzy_0.941
8,PETROMEDITERRÁNEA,SUPPLIER_043,fuzzy_0.941
9,PETROMEDITERRANEA,SUPPLIER_043,fuzzy_0.941


### Detail rebuild summary

,detail_rows_output,detail_unique_day_keys,detail_keys_used_for_v2_rebuild
0,197370,68961,3281


### Heuristica same-month y lookahead

,heuristic_unique_keys,heuristic_unique_keys_with_lookahead,heuristic_v2_rows_with_lookahead
0,546,182,887


### Muestra de claves reconstruidas por heuristica

,fecha_evento,producto_canonico,proveedor_candidato,transport_bucket_origin,transport_days_gap,transport_lookahead_flag,source_date_for_transport
145,2027-01-04,PRODUCT_002,SUPPLIER_008,heuristic_same_month_only,5,1,2027-01-09
180,2027-01-04,PRODUCT_003,SUPPLIER_039,heuristic_same_month_only,23,1,2027-01-27
191,2027-01-04,PRODUCT_001,SUPPLIER_008,heuristic_same_month_only,5,1,2027-01-09
409,2027-01-09,PRODUCT_002,SUPPLIER_039,heuristic_same_month_only,2,1,2027-01-11
416,2027-01-09,PRODUCT_003,SUPPLIER_039,heuristic_same_month_only,18,1,2027-01-27
601,2027-01-11,PRODUCT_003,SUPPLIER_039,heuristic_same_month_only,16,1,2027-01-27
772,2027-01-12,PRODUCT_001,SUPPLIER_039,heuristic_same_month_only,1,0,2027-01-11
827,2027-01-12,PRODUCT_002,SUPPLIER_039,heuristic_same_month_only,1,0,2027-01-11
1034,2027-01-12,PRODUCT_003,SUPPLIER_039,heuristic_same_month_only,15,1,2027-01-27
1304,2027-01-13,PRODUCT_003,SUPPLIER_039,heuristic_same_month_only,14,1,2027-01-27


## 4. Calidad del artefacto reconstruido y datasets Day 04.2

El builder Day 04.2 genera dos datasets auditables:
- `V2 + transport_rebuilt_only`
- `V2 + dispersion + transport_rebuilt`

Antes de entrenar, aplica dos gates:
- cobertura minima train/test `>= 0.80`,
- ausencia de leakage por `lookahead` en la heuristica.

In [6]:
display(Markdown("### Gate Day 04.2"))
display(pd.DataFrame([day042_quality["gate"]]))

display(Markdown("### Stage coverage Day 04.2"))
display(pd.DataFrame(day042_quality["stage_coverage"]).T.reset_index().rename(columns={"index": "stage"}))

display(Markdown("### Variantes construidas"))
variant_rows = []
for variant_name, payload in day042_quality["variants"].items():
    variant_rows.append({
        "variant_name": variant_name,
        "dataset_path": payload["dataset_path"],
        "rows_output": payload["rows_output"],
        "events_output": payload["events_output"],
        "invalid_positive_events": payload["events_with_invalid_positive_count"],
        "added_columns": len(payload["added_columns"]),
    })
display(pd.DataFrame(variant_rows))


### Gate Day 04.2

,coverage_gate_pass,leakage_gate_pass,structural_gate_pass,training_allowed,coverage_train_final,coverage_test_final,coverage_train_final_no_lookahead,coverage_test_final_no_lookahead,failure_reasons
0,True,False,True,False,0.892423,0.983743,0.885968,0.983023,[heuristic_lookahead_detected]


### Stage coverage Day 04.2

,stage,coverage_train,coverage_test
0,raw_explicit,0.866663,0.974966
1,parser_fix_plus_deterministic,0.870794,0.974966
2,final_after_heuristic,0.892423,0.983743
3,final_after_heuristic_no_lookahead,0.885968,0.983023


### Variantes construidas

,variant_name,dataset_path,rows_output,events_output,invalid_positive_events,added_columns
0,transport_rebuilt_only,data/public/day042/dataset_modelo_v2_transport...,155946,15404,0,17
1,dispersion_plus_transport_rebuilt,data/public/day042/dataset_modelo_v2_dispersio...,155946,15404,0,22


## 5. Metricas de `transport_only` y `transport + dispersion`

En Day 04.2 la respuesta correcta puede ser perfectamente `no_train`. Si el gate tecnico cae antes del modelado, no tiene sentido abrir metrica de modelo que no seria comparable ni defendible.

In [7]:
display(Markdown("### Resumen de training Day 04.2"))
display(pd.DataFrame([day042_summary]))

if day042_summary["status"] == "gated_no_train":
    display(Markdown(
        f"**Decision tecnica:** no se entrena. Reasons = `{day042_summary['gate']['failure_reasons']}`; "
        f"coverage_train_final = `{day042_summary['gate']['coverage_train_final']:.6f}`; "
        f"coverage_train_final_no_lookahead = `{day042_summary['gate']['coverage_train_final_no_lookahead']:.6f}`."
    ))
else:
    display(Markdown("Day 04.2 si llego a entrenar; revisar metrics json y registry oficial."))


### Resumen de training Day 04.2

,status,run_id,ts_utc,day_id,gate,baseline_metrics,trained_variants,policy_variant,message
0,gated_no_train,20260306T_day042_train,2026-03-06T10:55:12.368284+00:00,Day 04.2,"{'coverage_gate_pass': False, 'leakage_gate_pa...","{'top2_hit': 0.862894, 'balanced_accuracy': 0....",[],,Day 04.2 no entrena porque el artefacto recons...


**Decision tecnica:** no se entrena. Reasons = `['coverage_below_min_threshold', 'heuristic_lookahead_detected']`; coverage_train_final = `0.780565`; coverage_train_final_no_lookahead = `0.771957`.

## 6. Decision final y abiertas para Day 05

Decision formal Day 04.2:
- baseline sigue como champion,
- la linea de transporte queda cerrada dentro del bloque Day 04,
- solo merece reabrirse si entra una nueva fuente historica o una feature de contexto/missingness que no dependa de heuristica con fuga temporal.

Abiertas razonables para Day 05:
- cerrar champion final y narrativa del producto,
- decidir si Brent (`V3_B`) merece iteracion aparte,
- evitar mezclar transporte reabierto con tuning de hiperparametros.